In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using aws nocodb


In [3]:
from birddog.database import Database
from birddog.database_updater import normalize_url

2026-08-14 11:14:28,704 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-08-14 11:14:28,830 [INFO] Translation is enabled. Using GCP translator
2026-08-14 11:14:28,831 [INFO] Using Google Cloud translation API
2026-08-14 11:14:28,831 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-08-14 11:14:29,945 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 11:14:30,155 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     4.82    39.00       0.00           24


In [5]:
all_doc_ids = db.get_all_ids("Documents")

2026-08-14 11:15:30,790 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   25.00     2.06    39.00       0.00           24
2026-08-14 11:16:31,056 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   31.00     1.41    39.00       0.00           24


In [6]:
len(all_doc_ids)

216830

In [15]:
doc_recs = db.read("Documents", all_doc_ids, fields=["url", "sha1_hash"])

2026-08-14 12:06:30,662 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   46.00     0.13    35.21       0.00           24
2026-08-14 12:07:30,711 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   51.00    16.95    38.60       0.00           24
2026-08-14 12:08:30,781 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   57.00    16.96    38.68       0.00           24


In [16]:
len(doc_recs)

216830

In [17]:
doc_recs[0]

{'Id': 272612,
 'url': 'https://www.szukajwarchiwach.gov.pl/de/seria?p_p_id=Seria&p_p_lifecycle=0&p_p_state=normal&p_p_mode=view&_Seria_nameofjsp=jednostki&_Seria_id_serii=587759',
 'sha1_hash': None}

In [10]:
def group_equiv_records(recs):
    result = {}
    for rec in recs:
        url = rec.get("url")
        n_url = normalize_url(url)
        entries = result.get(n_url, [])
        entries.append(rec)
        result[n_url] = entries
    return result

In [11]:
grouped_doc_recs = group_equiv_records(doc_recs)

In [12]:
def select_aliased_doc_recs(groups):
    return {k: v for k, v in groups.items() if len(v) > 1}

In [13]:
aliased_doc_recs = select_aliased_doc_recs(grouped_doc_recs)

In [14]:
len(aliased_doc_recs)

0

In [ ]:
list(aliased_doc_recs.items())[0]

In [ ]:
def field_list(db, table_name="Documents"):
    recs = db.scan_all(
        "Schema", 
        where=("table_name", "eq", table_name), 
        fields="field_name")
    return [rec["field_name"] for rec in recs]

In [ ]:
doc_fields = field_list(db)

In [ ]:
def fold_field(recs, primary_rec, field_name):
    value = primary_rec.get(field_name)
    if value:
        return value
    for r in recs:
        value = r.get(field_name)
        if value:
            return value
    return primary_rec.get(field_name)    

In [ ]:
def fold_aliased_records(db, recs, fields=doc_fields):
    if len(recs) < 2:
        return recs[:1]
    n_url = normalize_url(recs[0]["url"])
    ids = [r["Id"] for r in recs]
    full_recs = db.read("Documents", ids, fields=fields)
    for rec in full_recs:
        if rec.get("url") == n_url:
            primary_rec = rec
            break
    result = primary_rec
    for field in fields:
        primary_rec[field] = fold_field(full_recs, primary_rec, field)
    return primary_rec

In [ ]:
folded_records = [
    fold_aliased_records(db, v)
    for v in aliased_doc_recs.values()
]

In [ ]:
len(folded_records)

In [ ]:
w_ids = db.write("Documents", folded_records)

In [ ]:
all([a == b["Id"] for a, b in zip(w_ids, folded_records)])

In [ ]:
def merge_page_links(db, aliased_recs):
    owners = []
    primary_id = None
    for r in aliased_recs:
        if r["url"] == normalize_url(r["url"]):
            primary_id = r["Id"]
        owners.extend(db.get_links("Documents", "owning_pages", r["Id"]))
    owners = list(set(owners))
    db.create_links("Documents", "owning_pages", primary_id, owners)
    return primary_id, owners

In [ ]:
merge_page_links(db, list(aliased_doc_recs.values())[0])

In [ ]:
for item in aliased_doc_recs.values():
    result = merge_page_links(db, item)
    print(result)

In [ ]:
def non_primaries(recs):
    return [r["Id"] for r in recs
            if r["url"] != normalize_url(r["url"])]

In [ ]:
np_ids = []
for v in aliased_doc_recs.values():
    np_ids.extend(non_primaries(v))
np_ids = list(set(np_ids))

In [ ]:
len(np_ids)

In [ ]:
np_ids

In [ ]:
db.delete("Documents", np_ids)

In [25]:
def group_records_by_hash(recs):
    result = {}
    for rec in recs:
        sha1 = rec.get("sha1_hash")
        if sha1:
            entries = result.get(sha1, [])
            entries.append(rec)
            result[sha1] = entries
    return result

In [26]:
docs_by_hash = group_records_by_hash(doc_recs)

In [27]:
len(docs_by_hash)

140656

In [28]:
dupe_docs_by_hash = {
    k: v for k, v in docs_by_hash.items()
    if len(v) > 1
}

In [29]:
len(dupe_docs_by_hash)

20182

In [32]:
list(dupe_docs_by_hash.items())[:5]

[('67895dcb51487c92659874248a60862026030e4b',
  [{'Id': 272613,
    'url': 'https://uk.wikisource.org/wiki/File:350-58_Inwentarz_wsi_klucza_starokonstantynowskiego.pdf',
    'sha1_hash': '67895dcb51487c92659874248a60862026030e4b'},
   {'Id': 272614,
    'url': 'https://commons.wikimedia.org/wiki/File:_350-58_Inwentarz_wsi_klucza_starokonstantynowskiego.pdf',
    'sha1_hash': '67895dcb51487c92659874248a60862026030e4b'}]),
 ('ab71cbec83341aec6f9282209847e142237e5c33',
  [{'Id': 272615,
    'url': 'https://commons.wikimedia.org/wiki/File:Інвентар_Степаня_та_околиць_з_1674_року.pdf',
    'sha1_hash': 'ab71cbec83341aec6f9282209847e142237e5c33'},
   {'Id': 272626,
    'url': 'https://uk.wikisource.org/wiki/File:Інвентар_Степаня_та_околиць_з_1674_року.pdf',
    'sha1_hash': 'ab71cbec83341aec6f9282209847e142237e5c33'},
   {'Id': 274717,
    'url': 'https://uk.wikisource.org/wiki/Файл:Інвентар_Степаня_та_околиць_з_1674_року.pdf',
    'sha1_hash': 'ab71cbec83341aec6f9282209847e142237e5c33'}]),
 

In [33]:
lengths = [len(v) for v in dupe_docs_by_hash.values()]

In [34]:
len(lengths)

20182

In [36]:
h = {}
for l in lengths:
    n = h.get(l, 0)
    h[l] = n + 1
h

{2: 19728, 3: 413, 4: 41}